# Phase 6 — Cross-Instrument Confirmation: DISPROVEN

The program's capstone: the FROZEN Phase-5 exit hypothesis, run with zero re-tuning over ~10 years
of ES, YM, and independent-vendor NQ 1-minute futures (17 walk-forward folds each), judged by a
pre-registered, git-timestamped 4-verdict decision table.

This notebook loads the committed `phase6_results.json` (it does **not** re-run the ~32-minute
single shot) and narrates the verdict. Full detail: `WRITEUP_PHASE6.md`.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

r = json.load(open("../phase6_results.json"))
print("VERDICT:", r["verdict_H_A"])
print("config_hash:", r["config_hash"][:16], "| git:", r["git_sha"][:10], "| run:", round(r["run_seconds"]), "s")

In [ ]:
c = r["conditions"]
rows = [
    ("pooled ES+YM net R-PF", round(c["pooled_r_pf"], 4)),
    ("day-cluster 90% CI", f"[{c['pooled_ci_lower']:.3f}, {c['pooled_ci_upper']:.3f}]"),
    ("CI upper < 1.0  ->  DISPROVEN clause (b)", c["pooled_ci_upper"] < 1.0),
    ("pooled n trades / trade-days", f"{c['pooled_n_trades']} / {c['pooled_n_days']}"),
    ("margin vs base: ES / YM", f"{c['margin_es']:+.3f} / {c['margin_ym']:+.3f} (bar: +0.10)"),
    ("indep-NQ agreement (tuned >= base)", c["nq_agreement"]),
    ("fold-majority tuned>base: ES / YM", f"{c['fold_majority_es']} / {c['fold_majority_ym']}"),
]
pd.DataFrame(rows, columns=["pre-registered quantity", "value"])

In [ ]:
tbl = []
for s in ("ES", "YM", "NQ"):
    b = r["instruments"][s]
    tbl.append({"instrument": s, "folds": b["n_folds"], "tuned n": b["tuned_n"],
                "tuned R-PF": round(b["tuned_r_pf"], 3), "base n": b["base_n"],
                "base R-PF": round(b["base_r_pf"], 3),
                "tuned net $": round(b["tuned_total_net_usd"]),
                "base net $": round(b["base_total_net_usd"])})
pd.DataFrame(tbl).set_index("instrument")

Both arms lose on every instrument. Exits *narrow* the loss on ES/YM (the Phase-4/5 story in
miniature) but cannot rescue a losing entry signal — and on clean NQ the tuned procedure is
**worse** than base. Both fixed-config replications (H-B1, H-B2) are "not supported" at the
Bonferroni gate.

In [ ]:
for k, v in r["hb_replication"].items():
    print(f"{k}: {v['replication']} | pooled R-PF {v['pooled_r_pf']:.3f} | "
          f"CI-low(2.5%) {v['ci_lower_bonferroni_2p5']:.3f} | margins {v['margins']}")

In [ ]:
s = r["sensitivity"]
cuts = {k: round(v, 3) for k, v in s.items() if k.endswith("_r_pf")}
pd.Series(cuts, name="tuned net R-PF").to_frame()

Unprofitable in every sensitivity slice — no hidden regime where it works.

In [ ]:
for p in ("phase6_verdict_matrix", "phase6_pooled_ci", "phase6_per_fold", "phase6_sensitivity"):
    img = mpimg.imread(f"../charts/{p}.png")
    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.imshow(img); ax.axis("off"); plt.show()


## Program epilogue

Six phases: data foundation → faithful rebuild (edge = selectivity) → tuning null → costs+exits
near-miss (failed its own significance gate) → **cross-instrument disproof on 10 years of cleaner
data** — plus, along the way, forensic proof of four defect classes in the original dataset.
Every headline number was pre-registered before it was observed. The deliverable is not a trading
edge; it is the demonstrated ability to find out, rigorously, whether one exists.